# Pandas 实用技巧

基础学完后的进阶技巧，实战中最常用。**先跑 Cell 1 建数据，再往下一个个跑。**

In [ ]:
# ===== 先建一份更真实的数据（带空值、重复值）=====
import pandas as pd
import numpy as np

df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六', '张三', '钱七', '孙八', '李四'],
    '年龄': [25, 30, 28, 35, 25, np.nan, 40, 30],
    '城市': ['北京', '上海', '北京', '深圳', '北京', '广州', '上海', '上海'],
    '工资': [8000, 12000, 9000, 15000, 8000, 11000, 20000, 12000],
})
df

## 技巧 1：查看数据 —— 不止 head()

In [ ]:
# df.head() 只看前5行，但你可能想知道更多
df.shape            # (8行, 4列) → 快速看数据大小
df.columns          # 看所有列名
df.dtypes           # 每列的数据类型（int64/float64/object）
df['姓名'].unique()    # 这一列有哪些不重复的值
df['姓名'].nunique()   # 不重复的值有几个
df['城市'].value_counts()  # 每个城市出现几次（超实用！看数据分布）

## 技巧 2：处理空值（np.nan）—— 真实数据一定有空值

In [ ]:
# 找空值
df.isnull()              # 整张表，True 表示是空值
df.isnull().sum()        # 每列有几个空值（最常用，一眼看出哪列缺数据）

In [ ]:
# 删掉有空值的行
df.dropna()              # 只要有空值的行全删掉
df.dropna(subset=['年龄'])  # 只看"年龄"列，这列为空的行才删

In [ ]:
# 填充空值（更常用，别轻易删数据）
df['年龄'].fillna(0)                    # 空值填 0
df['年龄'].fillna(df['年龄'].mean())    # 空值填平均年龄（最常见做法）
df['年龄'].fillna(df['年龄'].median())  # 空值填中位数（比平均更抗异常值）

## 技巧 3：去重 —— 数据里常有重复记录

In [ ]:
# 看有没有重复
df.duplicated()          # True 表示这行跟前面某行完全一样
df.duplicated().sum()    # 有几行是重复的

In [ ]:
# 去重
df.drop_duplicates()                    # 完全相同的行去重
df.drop_duplicates(subset=['姓名'])      # 只按姓名去重（同名就算重复，保留第一个）
df.drop_duplicates(subset=['姓名'], keep='last')  # 同名保留最后一个

## 技巧 4：apply —— 对每一行/每个值做自定义操作（最灵活）

In [ ]:
# 对一列的每个值做处理
# 比如给工资加个"元"字
df['工资'].apply(lambda x: f'{x}元')    # 每个值都变成 "8000元" 这种格式

In [ ]:
# 更实用：根据工资分等级
def grade(salary):
    if salary >= 15000:
        return '高薪'
    elif salary >= 10000:
        return '中薪'
    else:
        return '普通'

df['工资'].apply(grade)    # 每个人的工资变成等级
# 真正用的时候要赋值：df['工资等级'] = df['工资'].apply(grade)

In [ ]:
# 对整行做操作（axis=1 表示按行）
# 比如算每个人的"工资/年龄"比值（每元年龄产出，瞎编的指标）
df.apply(lambda row: row['工资'] / row['年龄'], axis=1)

## 技巧 5：map —— 简单的值替换

In [ ]:
# 把城市名换成英文（一一对应）
city_map = {'北京': 'Beijing', '上海': 'Shanghai', '深圳': 'Shenzhen', '广州': 'Guangzhou'}
df['城市'].map(city_map)    # 每个城市名换成对应英文

## 技巧 6：如果...否则...（类似 SQL 的 CASE WHEN）

In [ ]:
# np.where(条件, 满足时的值, 不满足时的值)
import numpy as np
df['是否高龄'] = np.where(df['年龄'] > 30, '是', '否')
df[['姓名', '年龄', '是否高龄']]

## 技巧 7：分组后多个统计量一次性算完

In [ ]:
# 按城市分组，同时看：人数、平均工资、最高工资、最低工资
df.groupby('城市').agg(
    人数=('姓名', 'count'),
    平均工资=('工资', 'mean'),
    最高工资=('工资', 'max'),
    最低工资=('工资', 'min'),
).round(0)    # round(0) 保留整数，去掉小数

In [ ]:
# 也可以对不同的列用不同的统计方式
df.groupby('城市').agg({
    '年龄': 'mean',       # 年龄取平均
    '工资': ['min', 'max'],  # 工资取最小和最大
})

## 技巧 8：合并两张表（merge）—— 相当于 SQL 的 JOIN

In [ ]:
# 假设有另一张表：每个人的部门
df_dept = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六'],
    '部门': ['技术部', '市场部', '技术部', '财务部'],
})
df_dept

In [ ]:
# 按"姓名"合并两张表（类似 SQL: df JOIN df_dept ON 姓名）
pd.merge(df, df_dept, on='姓名', how='left')
# how='left'  → 以左表为准，右表没有的填空值
# how='inner' → 两边都有的才保留（默认）
# how='outer' → 全保留，没有的填空值

## 技巧 9：列名重命名

In [ ]:
# 把列名改成英文（方便后续操作，中文列名有时候会有小问题）
df.rename(columns={
    '姓名': 'name',
    '年龄': 'age',
    '城市': 'city',
    '工资': 'salary',
})

## 技巧 10：链式写法 —— 代码更干净

In [ ]:
# 一步步做也行，但链式写法更简洁：
# 筛选北京的人 → 按工资降序 → 只看姓名和工资
(df[df['城市'] == '北京']
   .sort_values('工资', ascending=False)
   [['姓名', '工资']])

# 每一步用括号包起来，换行写，读起来很清晰

## 技巧 11：透视表 pivot_table —— 相当于 Excel 透视表

In [ ]:
# 按城市看平均工资（行=城市，值=平均工资）
pd.pivot_table(df, values='工资', index='城市', aggfunc='mean').round(0)

In [ ]:
# 二维透视表：行=城市，列=看有没有高龄，值=平均工资
df['高龄'] = np.where(df['年龄'] > 30, '是', '否')
pd.pivot_table(df, values='工资', index='城市', columns='高龄', aggfunc='mean').round(0)

## 技巧 12：读取真实数据时的小技巧

In [ ]:
# 读 CSV 时常用参数（这里演示，没文件不会真跑）
# 读文件只读前 100 行先看看（大文件别一次全读）
# pd.read_csv('data.csv', nrows=100)

# 指定哪几列读（省内存）
# pd.read_csv('data.csv', usecols=['姓名', '工资'])

# 指定某列为索引
# pd.read_csv('data.csv', index_col='姓名')

# 中文文件乱码时试试
# pd.read_csv('data.csv', encoding='gbk')